# <font color="steelblue">09. Medidas de posición: cuartiles, percentiles y diagrama de caja</font>

*Unidad 2. Estadística descriptiva* · Sesión teórica 9 de 30

**Autoría**:

*   Fernando Borrás Rocher (f.borras@umh.es), autor de la idea original
*   Juan Francisco Monge Ivars
*   José Santiago García Cremades

Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

Basado en [SIMUSTAT](https://unidigitalsimustat.umh.es/) (proyecto UNIDIGITAL-UMH, CC BY-SA 4.0).

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`salarios_redaccion.csv`, `audiencias_tv.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["salarios_redaccion.csv", "audiencias_tv.csv"]
PREGUNTAS = ["q_P09_1.json", "q_P09_2.json", "q_P09_3.json", "q_P09_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

«El 10 % de la plantilla que más cobra gana más que el 50 % que menos.» Esa frase usa **percentiles**. Las medidas de posición dicen dónde queda un valor respecto al resto y son la base del **diagrama de caja**, el mejor gráfico para comparar varios grupos de un vistazo.

**Qué necesitas saber antes**: el cuaderno 8 (mediana).

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Calcular e interpretar cuartiles, deciles y percentiles.
2. Situar un valor concreto en la distribución (su percentil).
3. Construir y leer un diagrama de caja.
4. Comparar grupos con diagramas de caja.

## <font color='steelblue'>Contenidos</font>

1. Cuartiles y percentiles
2. ¿En qué posición está un valor?
3. El diagrama de caja
4. Comparar grupos con cajas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Cuartiles y percentiles</font>

In [ ]:
salarios = pd.read_csv("salarios_redaccion.csv")
audiencias = pd.read_csv("audiencias_tv.csv")
s = salarios["salario_bruto_anual"]

El **percentil p** es el valor que deja por debajo el p % de los datos. Algunos tienen nombre propio:

- **Cuartiles**: $Q_1$ (percentil 25), $Q_2$ (percentil 50 = mediana) y $Q_3$ (percentil 75).
- **Deciles**: percentiles 10, 20, ..., 90.

In [ ]:
print(s.quantile([0.25, 0.5, 0.75]))

**Cómo leerlo**: un cuarto de la plantilla cobra menos de $Q_1$ y otro cuarto más de $Q_3$. La mitad central está entre ambos.

In [ ]:
deciles = s.quantile([0.1, 0.9])
print(deciles)
print("Ratio P90/P10:", round(deciles[0.9] / deciles[0.1], 2))

El **ratio P90/P10** es un indicador de desigualdad muy usado en informes de salarios y renta: cuántas veces más cobra quien está en el percentil 90 que quien está en el 10.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P09_1.json")

## <font color='steelblue'>2. ¿En qué posición está un valor?</font>

También podemos hacer la pregunta inversa: ¿qué percentil ocupa un sueldo de 30.000 euros? Basta con calcular qué proporción de datos está por debajo.

In [ ]:
valor = 30000
percentil = (s < valor).mean() * 100
print(f"Un salario de {valor} € está en el percentil {percentil:.0f}")

Es la forma correcta de contar en una noticia si un dato es alto o bajo: «cobra más que el 70 % de sus compañeros» se entiende mejor que cualquier media.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P09_2.json")

## <font color='steelblue'>3. El diagrama de caja</font>

El **diagrama de caja** (*boxplot*) resume cinco números en un dibujo:

- la **caja** va de $Q_1$ a $Q_3$ y la raya interior es la **mediana**;
- los **bigotes** llegan hasta el dato más extremo que no se aleja más de 1,5 veces el rango intercuartílico ($RIC = Q_3 - Q_1$);
- los puntos sueltos son **valores atípicos** (lo verás en el cuaderno 11).

In [ ]:
plt.boxplot(s, vert=False)
plt.xlabel("Salario bruto anual (€)")
plt.title("Salarios de la redacción (datos simulados)")
plt.show()

**Cómo leerlo**: la mediana está más cerca de $Q_1$ que de $Q_3$ y el bigote derecho es más largo: confirma la asimetría a la derecha. Los puntos de la derecha son los sueldos de dirección.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P09_3.json")

## <font color='steelblue'>4. Comparar grupos con cajas</font>

**Qué vamos a hacer**: comparar el salario por tipo de medio. Con Pandas basta `boxplot(column=..., by=...)`.

In [ ]:
salarios.boxplot(column="salario_bruto_anual", by="tipo_medio", vert=False, grid=False, figsize=(10, 4))
plt.suptitle("")
plt.title("Salario según tipo de medio (datos simulados)")
plt.xlabel("Salario bruto anual (€)")
plt.show()

En un solo gráfico se comparan cinco grupos: dónde está su centro (mediana), cuánto varían (tamaño de la caja) y si tienen casos extremos. Hacer lo mismo con cinco histogramas sería mucho más difícil.

In [ ]:
audiencias.boxplot(column="share", by="franja", grid=False, figsize=(10, 4))
plt.suptitle("")
plt.title("Share por franja horaria (datos simulados)")
plt.ylabel("Share (%)")
plt.show()

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P09_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Calcula los cuartiles del share de los programas de televisión.

<details><summary>Ver solución</summary>

```python
audiencias["share"].quantile([0.25, 0.5, 0.75])
```
</details>

### **Ejercicio 2** *(sección 1)*

Calcula el ratio P90/P10 de los salarios **solo de redactores y redactoras**. ¿Es mayor o menor que el de toda la plantilla? ¿Por qué?

<details><summary>Ver solución</summary>

```python
r = salarios.loc[salarios["puesto"] == "Redactor/a", "salario_bruto_anual"]
r.quantile(0.9) / r.quantile(0.1)
```

Menor: dentro de un mismo puesto los salarios son más parecidos.
</details>

### **Ejercicio 3** *(sección 2)*

Un programa tiene 1.900 miles de espectadores. ¿En qué percentil está dentro de todos los programas?

<details><summary>Ver solución</summary>

```python
round((audiencias["espectadores_miles"] < 1900).mean() * 100)
```
</details>

### **Ejercicio 4** *(sección 4)*

Haz un diagrama de caja del salario según el tipo de contrato. ¿Qué grupo tiene la mediana más baja?

<details><summary>Ver solución</summary>

```python
salarios.boxplot(column="salario_bruto_anual", by="contrato", vert=False, grid=False)
plt.show()
```
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- El **percentil p** deja por debajo el p % de los datos; los **cuartiles** son los percentiles 25, 50 y 75.
- El **ratio P90/P10** mide la desigualdad entre la parte alta y la baja de una distribución.
- Para situar un valor: `(serie < valor).mean() * 100` da su percentil.
- El **diagrama de caja** resume mediana, cuartiles, bigotes y atípicos, y es el mejor gráfico para **comparar grupos**.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Percentil** | Valor que deja por debajo un porcentaje dado de los datos. |
| **Cuartil** | Percentiles 25, 50 y 75. |
| **Rango intercuartílico (RIC)** | Q3 − Q1: amplitud de la mitad central. |
| **Diagrama de caja** | Gráfico que resume la distribución con cinco números. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| Aparece un título duplicado encima del boxplot | Pandas añade un título por grupo | Usa `plt.suptitle("")` |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 10 medirás la dispersión: rango, varianza, desviación típica y coeficiente de variación.

**Siguiente cuaderno**: [10. Medidas de dispersión: ¿son los datos parecidos o muy distintos?](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P10_medidas_de_dispersion.ipynb)